In [ ]:
import pandas as pd

In [ ]:
# Flattening nested JSON structures using pandas:

raw_events = [
    {"event_id": "e_01", "timestamp": "2026-11-12T08:00:00Z", "actor": {"actor_id": 101, "actor_role": "admin"}},
    {"event_id": "e_02", "timestamp": "2026-11-12T08:05:00Z", "actor": {"actor_id": 102, "actor_role": "viewer"}},
]

# Flatten the nested structure
df_flattened = pd.json_normalize(raw_events)

# Rename the columns to remove the nested prefixes for clarity
df_flattened.columns = [col.split(".")[-1] for col in df_flattened.columns]

print("\nCleaned columns:")
print(df_flattened)

# Now, let's demonstrate how to flatten a list of dictionaries that contain nested lists, which is a common scenario in order processing systems.

orders = [
    {
        "order_id": 501, 
        "customer": "Acme Corp", 
        "items": [
            {"sku": "A1", "qty": 2}, 
            {"sku": "B2", "qty": 5}
        ]
    },
    {
        "order_id": 502, 
        "customer": "Globex", 
        "items": [
            {"sku": "C3", "qty": 1}
        ]
    }
]

# Flatten the 'items' list into rows, while keeping 'order_id' and 'customer' attached
df_items = pd.json_normalize(
    orders,
    record_path=["items"],           # The list we are exploding into rows
    meta=["order_id", "customer"],    # Parent fields to repeat for context
    record_prefix = "items_"          # Prefix for the new columns to avoid name clashes
)

print(df_items)

In [ ]:
# Data Validation Functions:

sales_data = {
    "order_id": [101, 102, 103, None, 101], # Contains a missing value and a duplicate order_id/item_id combo
    "item_id": ["A", "B", "C", "A", "A"],
    "amount": [50.0, 120.0, -15.0, 45.0, 50.0] # Contains a negative amount
}

df_sales = pd.DataFrame(sales_data)


# Validation function with consolidated error reporting and count of errors:
def validate_dataset_report(df):
    errors = []
    
    # 1. Count nulls in order_id
    null_count = df["order_id"].isnull().sum()
    if null_count > 0:
        errors.append(f"order_id has nulls in {null_count} row(s)")
    
    # 2. Count negative amounts
    neg_count = (df["amount"] < 0).sum()
    if neg_count > 0:
        errors.append(f"amount has negative values in {neg_count} row(s)")
        
    # 3. Count duplicate composite keys
    dup_count = df.duplicated(subset=["order_id", "item_id"]).sum()
    if dup_count > 0:
        errors.append(f"duplicate order_id/item_id combinations in {dup_count} row(s)")
        
    # If any errors were logged, raise a consolidated exception with the count
    if errors:
        error_summary = f"Data Quality Validation Failed! Found {len(errors)} error category(ies):\n" + "\n".join(f"  - {err}" for err in errors)
        raise ValueError(error_summary)
        
    print("Validation passed successfully! No data quality errors found.")


# Validation function with immediate error raising:
# Note that .any() is used to check if any of the conditions are True across the entire DataFrame, which is more efficient than checking each row individually.
# Note: the inverse of .any() is .all(), which checks if all values meet a condition.    
def validate_dataset(df):
    # 1. Check for nulls in order_id
    if df["order_id"].isnull().any():
        raise ValueError("Data Quality Error: order_id contains null values.")
    
    # 2. Check for negative amounts
    if (df["amount"] < 0).any():
        raise ValueError("Data Quality Error: amount contains negative values.")
        
    # 3. Check for duplicate composite keys (order_id + item_id)
    if df.duplicated(subset=["order_id", "item_id"]).any():
        raise ValueError("Data Quality Error: duplicate order_id and item_id combinations found.")
        
    print("Validation passed successfully! No data quality errors found.")


print(validate_dataset_report(df_sales))
print(validate_dataset(df_sales))
